In [13]:
from ucimlrepo import fetch_ucirepo
import numpy as np
import pickle
import copy
import pandas as pd
from sklearn import preprocessing

In [2]:
# fetch dataset 
adult = fetch_ucirepo(id=2) 
  
# data (as pandas dataframes) 
X = adult.data.features 
y = adult.data.target

In [32]:
X.columns

Index(['age', 'workclass', 'fnlwgt', 'education', 'education-num',
       'marital-status', 'occupation', 'relationship', 'race', 'sex',
       'capital-gain', 'capital-loss', 'hours-per-week', 'native-country'],
      dtype='object')

In [4]:
# X.iloc[27]

In [5]:
def map_array_values(array, value_map):
    # value map must be { src : target }
    ret = array.copy()
    for src, target in value_map.items():
        ret[ret == src] = target
    return ret

feature_names = ["Age", "Workclass", "fnlwgt", "Education",
                         "Education-Num", "Marital Status", "Occupation",
                         "Relationship", "Race", "Sex", "Capital Gain",
                         "Capital Loss", "Hours per week", "Country", 'Income']
features_to_use = [0, 1, 3, 5, 6, 7, 8, 9, 10, 11, 12, 13]
categorical_features = [1, 3, 5, 6, 7, 8, 9, 10, 11, 13]
education_map = {
    '10th': 'Dropout', '11th': 'Dropout', '12th': 'Dropout', '1st-4th':
    'Dropout', '5th-6th': 'Dropout', '7th-8th': 'Dropout', '9th':
    'Dropout', 'Preschool': 'Dropout', 'HS-grad': 'High School grad',
    'Some-college': 'High School grad', 'Masters': 'Masters',
    'Prof-school': 'Prof-School', 'Assoc-acdm': 'Associates',
    'Assoc-voc': 'Associates',
}
occupation_map = {
    "Adm-clerical": "Admin", "Armed-Forces": "Military",
    "Craft-repair": "Blue-Collar", "Exec-managerial": "White-Collar",
    "Farming-fishing": "Blue-Collar", "Handlers-cleaners":
    "Blue-Collar", "Machine-op-inspct": "Blue-Collar", "Other-service":
    "Service", "Priv-house-serv": "Service", "Prof-specialty":
    "Professional", "Protective-serv": "Other", "Sales":
    "Sales", "Tech-support": "Other", "Transport-moving":
    "Blue-Collar",
}
country_map = {
    'Cambodia': 'SE-Asia', 'Canada': 'British-Commonwealth', 'China':
    'China', 'Columbia': 'South-America', 'Cuba': 'Other',
    'Dominican-Republic': 'Latin-America', 'Ecuador': 'South-America',
    'El-Salvador': 'South-America', 'England': 'British-Commonwealth',
    'France': 'Euro_1', 'Germany': 'Euro_1', 'Greece': 'Euro_2',
    'Guatemala': 'Latin-America', 'Haiti': 'Latin-America',
    'Holand-Netherlands': 'Euro_1', 'Honduras': 'Latin-America',
    'Hong': 'China', 'Hungary': 'Euro_2', 'India':
    'British-Commonwealth', 'Iran': 'Other', 'Ireland':
    'British-Commonwealth', 'Italy': 'Euro_1', 'Jamaica':
    'Latin-America', 'Japan': 'Other', 'Laos': 'SE-Asia', 'Mexico':
    'Latin-America', 'Nicaragua': 'Latin-America',
    'Outlying-US(Guam-USVI-etc)': 'Latin-America', 'Peru':
    'South-America', 'Philippines': 'SE-Asia', 'Poland': 'Euro_2',
    'Portugal': 'Euro_2', 'Puerto-Rico': 'Latin-America', 'Scotland':
    'British-Commonwealth', 'South': 'Euro_2', 'Taiwan': 'China',
    'Thailand': 'SE-Asia', 'Trinadad&Tobago': 'Latin-America',
    'United-States': 'United-States', 'Vietnam': 'SE-Asia'
}
married_map = {
    'Never-married': 'Never-Married', 'Married-AF-spouse': 'Married',
    'Married-civ-spouse': 'Married', 'Married-spouse-absent':
    'Separated', 'Separated': 'Separated', 'Divorced':
    'Separated', 'Widowed': 'Widowed'
}

def cap_gains_fn(x):
    x = x.astype(float)
    d = np.digitize(x, [0, np.median(x[x > 0]), float('inf')],
                    right=True).astype(str)
    return map_array_values(d, {'0': 'No gains', '1': 'Low gains', '2': 'High gains'})

def cap_loss_fn(x):
    x = x.astype(float)
    d = np.digitize(x, [0, np.median(x[x > 0]), float('inf')],
                    right=True).astype(str)
    return map_array_values(d, {'0': 'No losses', '1': 'Low losses', '2': 'High losses'})

transformations = {
    3: lambda x: map_array_values(x, education_map),
    5: lambda x: map_array_values(x, married_map),
    6: lambda x: map_array_values(x, occupation_map),
    10: cap_gains_fn,
    11: cap_loss_fn,
    13: lambda x: map_array_values(x, country_map),
}

In [6]:
# import pickle
# X_previous = pickle.load(open(r"C:\Users\Louth\Downloads\Hase_Bansal\tabular\data\adult\train\data.pkl", "rb"))

In [7]:
# sum(X_previous==-1)

In [8]:
X_numpy = X.values

In [9]:
for feature, fun in transformations.items():
        X_numpy[:, feature] = fun(X_numpy[:, feature])

In [27]:
y_numpy = y.values[:, 0]
for i, label in enumerate(y_numpy):
    if label[-1]==".":
            y_numpy[i] = label[:-1]

In [30]:
# Shortlist only the feature we want to use
X_numpy = X_numpy[:, features_to_use]
feature_names = ([x for i, x in enumerate(feature_names)
                  if i in features_to_use])
categorical_features = ([features_to_use.index(x)
                             for x in categorical_features])

In [25]:
# This cell converts the data from strings into numerical data by counting all the labels
categorical_names = {}
for feature in categorical_features:
    le = sklearn.preprocessing.LabelEncoder()
    le.fit(X_numpy[:, feature])
    data[:, feature] = le.transform(data[:, feature])
    categorical_names[feature] = le.classes_
data = data.astype(float)

array([1, 1, 1, 1, 3])

In [ ]:
ordinal_features = []
discretize=True

if discretize:
    disc = lime.lime_tabular.QuartileDiscretizer(data,
                                                 categorical_features,
                                                 feature_names)
    data = disc.discretize(data)
    ordinal_features = [x for x in range(data.shape[1])
                        if x not in categorical_features]
    categorical_features = range(data.shape[1])
    categorical_names.update(disc.names)

In [28]:
# Similar to the X above, we can also preprocess the y to be a numerical value rather than a string
le = sklearn.preprocessing.LabelEncoder()
le.fit(y_numpy)
y_numerical = le.transform(y_numpy)
y_class_names = list(le.classes_)
class_target = "Income"